Preprocess Input Sequences: ATCGM → One-hot → Embedding

In [23]:
import torch
import torch.nn as nndeactivate
import torch.nn as nn

In [24]:
import torch
print(torch.__file__)
print(torch.__version__)
print(torch.nn.Module)

/Users/katebarouch/torch-env/lib/python3.13/site-packages/torch/__init__.py
2.7.0
<class 'torch.nn.modules.module.Module'>


In [25]:
# Vocabulary: A, T, C, G, M
vocab = ['A', 'T', 'C', 'G', 'M']
vocab_dict = {base: idx for idx, base in enumerate(vocab)}

class DNAMethylationTokenizer:
    def __init__(self, vocab_dict):
        self.vocab_dict = vocab_dict

    def tokenize(self, sequence):
        return [self.vocab_dict[base] for base in sequence]

tokenizer = DNAMethylationTokenizer(vocab_dict)
sequence = "ATCMG"
token_ids = tokenizer.tokenize(sequence)  # [0, 1, 2, 4, 3]

token_tensor = torch.tensor(token_ids).unsqueeze(0)  # (batch_size, seq_len)

Token + Positional Embedding

In [26]:
class MethylationEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, max_seq_len):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embedding = nn.Parameter(self._init_sinusoidal(max_seq_len, embed_dim), requires_grad=False)

    def _init_sinusoidal(self, max_len, d_model):
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # (1, max_len, d_model)

    def forward(self, x):
        seq_len = x.size(1)
        return self.token_embedding(x) + self.pos_embedding[:, :seq_len, :]

# Usage
embedding_layer = MethylationEmbedding(vocab_size=5, embed_dim=768, max_seq_len=512)
embedded = embedding_layer(token_tensor)

Transformer Encoder (L Layers)

In [27]:
transformer_layer = nn.TransformerEncoderLayer(
    d_model=768,
    nhead=12,  # Number of attention heads
    dim_feedforward=3072,
    activation='gelu',
    batch_first=True  # Makes input shape (batch, seq_len, d_model)
)

transformer_encoder = nn.TransformerEncoder(transformer_layer, num_layers=12)
encoded_output = transformer_encoder(embedded)

Task-Specific Heads

In [28]:
class TaskHeads(nn.Module):
    def __init__(self, hidden_dim, num_tissues):
        super().__init__()
        self.tissue_head = nn.Linear(hidden_dim, num_tissues)       # Softmax later
        self.expression_head = nn.Linear(hidden_dim, 1)             # Regression
        self.methylation_head = nn.Linear(hidden_dim, 1)            # Sigmoid

    def forward(self, x, cls_index=0):
        # For simplicity, use CLS token (position 0) for tissue and expression
        cls_token = x[:, cls_index, :]  # (batch, hidden_dim)
        tissue_logits = self.tissue_head(cls_token)
        expression_pred = self.expression_head(cls_token)

        # Methylation prediction for each position
        methylation_pred = torch.sigmoid(self.methylation_head(x))  # (batch, seq_len, 1)
        return tissue_logits, expression_pred, methylation_pred

Putting It All Together in One Model

In [29]:
class MethylationTransformer(nn.Module):
    def __init__(self, vocab_size=5, embed_dim=768, seq_len=512, num_layers=12, num_heads=12, ff_dim=3072, num_tissues=10):
        super().__init__()
        self.embedding = MethylationEmbedding(vocab_size, embed_dim, seq_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            activation='gelu',
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.heads = TaskHeads(embed_dim, num_tissues)

    def forward(self, x):
        x = self.embedding(x)            # (batch, seq_len, embed_dim)
        x = self.encoder(x)              # (batch, seq_len, embed_dim)
        return self.heads(x)             # tissue, expression, methylation

Instantiate the Model

In [30]:
model = MethylationTransformer(
    vocab_size=5,         # A, T, C, G, M
    embed_dim=768,
    seq_len=512,
    num_layers=12,
    num_heads=12,
    ff_dim=3072,
    num_tissues=10        # for example: liver, lung, brain, etc.
)

Prepare Input

In [31]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

Create Fake Data

In [32]:
import torch
import random
from sklearn.model_selection import train_test_split

# Set seed for reproducibility
random.seed(42)
torch.manual_seed(42)

vocab_dict = {'A': 0, 'T': 1, 'C': 2, 'G': 3, 'M': 4}
tokenizer = DNAMethylationTokenizer(vocab_dict)

def generate_fake_sequence(length=500):
    bases = list(vocab_dict.keys())
    return ''.join(random.choices(bases, k=length))

# Generate 10 fake samples
sequences = [generate_fake_sequence() for _ in range(10)]
input_tensors = [torch.tensor(tokenizer.tokenize(seq)) for seq in sequences]  # shape (seq_len,)
input_tensors = torch.stack(input_tensors)  # shape (10, 500)

# Fake labels
y_tissue = torch.randint(0, 3, (10,))             # 3 tissue classes
y_expr = torch.randn(10, 1)                       # random expression levels
y_methyl = torch.randint(0, 2, (10, 500)).float() # binary methylation per position


In [34]:
# 1. Number of samples
num_samples = input_tensors.shape[0]  # e.g. 10
split_idx = int(0.7 * num_samples)    # 70% for training

# 2. Shuffle indices
perm = torch.randperm(num_samples)
train_idx = perm[:split_idx]
test_idx = perm[split_idx:]

# 3. Apply indices to tensors
X_train = input_tensors[train_idx]
X_test = input_tensors[test_idx]
y_tissue_train = y_tissue[train_idx]
y_tissue_test = y_tissue[test_idx]
y_expr_train = y_expr[train_idx]
y_expr_test = y_expr[test_idx]
y_methyl_train = y_methyl[train_idx]
y_methyl_test = y_methyl[test_idx]

Loss Functions

tissue_logits: shape (batch_size, num_tissues) — softmax for classification

expression_pred: shape (batch_size, 1) — linear output

methylation_pred: shape (batch_size, seq_len, 1) — sigmoid for each base

In [38]:
def compute_loss(tissue_logits, expr_pred, methyl_pred, y_tissue, y_expr, y_methyl, return_parts=False):
    loss_fn_tissue = nn.CrossEntropyLoss()
    loss_fn_expr = nn.MSELoss()
    loss_fn_methyl = nn.BCELoss()

    tissue_loss = loss_fn_tissue(tissue_logits, y_tissue)
    expr_loss = loss_fn_expr(expr_pred, y_expr)
    methyl_loss = loss_fn_methyl(methyl_pred.squeeze(-1), y_methyl)

    total_loss = tissue_loss + expr_loss + methyl_loss

    if return_parts:
        return total_loss, tissue_loss.item(), expr_loss.item(), methyl_loss.item()
    else:
        return total_loss

Train Step

In [39]:
model.train()
optimizer.zero_grad()

tissue_logits, expr_pred, methyl_pred = model(X_train)
loss, l_tissue, l_expr, l_methyl = compute_loss(
    tissue_logits, expr_pred, methyl_pred,
    y_tissue_train, y_expr_train, y_methyl_train,
    return_parts=True
)

loss.backward()
optimizer.step()

print(f"Total loss: {loss.item():.4f} | Tissue: {l_tissue:.4f} | Expression: {l_expr:.4f} | Methylation: {l_methyl:.4f}")

Total loss: 22.5952 | Tissue: 1.6378 | Expression: 20.1940 | Methylation: 0.7635
